In [1]:
import os
import pickle
import dill
import pprint
import itertools
import pathos
import pprint
import functools
from functools import partial
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats as st

import plotly.graph_objects as go
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

import sys
sys.path.append('/Users/leonardo.tessarolo/git/bayesian_bss/')

from src import MMSEMetropolisHastingsEstimator, MAPGradientAscentEstimator, InstantaneousMixtureModel, PosteriorContourLines, ContourLineGraphPlotter, MCMCGraphPlotter, MAPGradientAscentGraphPlotter, ExponentialPrior, LogisticSource, BayesianEstimators, TriangularSource, ExperimentExecutor

print(os.cpu_count())

np.random.seed(2000)

8


In [2]:
# TODO: criar descritivo I/O geral das simulações

In [3]:
# Whether or not to create config
CREATE_CONFIG=False

# Whether or not to create folder structure
CREATE_FOLDER_STRUCTURE=False

# Experiment name
EXPERIMENT_NAME='test_async_io'

# Folder which will contain output directory tree
OUTPUT_DIR='./output'
base_output_path=Path(OUTPUT_DIR)

# Create experiment directory
experiment_dir = base_output_path / EXPERIMENT_NAME

# 1. Configurations

In [4]:
if CREATE_CONFIG:
    # Number of sources and observations
    NSOURCES=2
    NOBS=1000

    # Mixing matrix configuration
    A = np.array([
        [1, 1],
        [-0.5, 0.5]
    ])

    # Define initial conditions for optimizations
    initial_B = np.linalg.inv(
        A
    ) + np.random.normal(
        0,0.1,
        A.shape
    )
    initial_B = initial_B.reshape(
        (NSOURCES, NSOURCES, 1)
    )

    # Execution configurations
    exec_configs = {
        'general': {
            # Experiment name for folder structure
            'experiment_name': EXPERIMENT_NAME,
            # Experiment directory
            'experiment_dir': experiment_dir,
            # Whether or not to create folder structure
            'create_folder_structure': CREATE_FOLDER_STRUCTURE,
            # Number of sources
            'n_sources': NSOURCES,
            # Number of observations in each realization
            'n_obs': NOBS,
            # Number of parallel workers
            'n_workers': os.cpu_count(),
            # Number of parallel realizations to run
            'n_realizations': 20,
            # Mixing matrix
            'A': A,
            # Separating matrix
            'B': np.linalg.inv(A),
            # Initial condition
            'initial_B': initial_B,
            # Whether or not to use a normalized posterior
            'normalize_posterior': True
        },
        'contour': {
            # Number of points used in contour line grids
            # 'contour_grid_points': 301,
            'contour_grid_points': 11,
            # Exploration limits for grids
            'u_lims':(-0.3, 0.3),
            'v_lims':(-0.3, 0.3)
        },
        'map': {
            # Threshold for stopping optimization
            'stopping_thresh': 0.0001,
            # Maximum number of iterations
            # 'max_it': 1000000,
            'max_it': 100,
            # 'max_it': 10,
            # Learning rate
            # 'learning_rate': 1E-4
            'learning_rate': 1E-3
        },
        'mcmc': {
            # Exploration variance for MCMC
            # 'exploration_var': 5E-5,
            'exploration_var': 5E-5,
            # Number of samples to generate
            # 'n_samples': 30000,
            'n_samples': 500,
            # 'n_samples':300,
            # Burn-in samples
            'burn_in': 0.5
        },
        'sim': {
            'source_model': LogisticSource(
                mu=0.0,
                sigma=1.0
            ),
            'sources': {
                'perfect_model': LogisticSource(
                    mu=0.0,
                    sigma=1.0
                ),
                'slightly_misspecified_model': LogisticSource(
                    mu=0.1,
                    sigma=1.1
                ),
                'largely_misspecified_model': TriangularSource(
                    lower=-2,
                    upper=2,
                    mode=0
                )
            },
            'priors': {
                'likelihood': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=np.inf
                ),
                'non_informative_prior': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=10
                ),
                'informative_prior': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=0.1
                ),
                'identity_transform': ExponentialPrior(
                    center=np.eye(NSOURCES),
                    std=0.1
                )
            },
            'test_cases': {
                'i': {
                    'source': 'perfect_model',
                    'prior': 'likelihood',
                },
                'ii': {
                    'source': 'perfect_model',
                    'prior': 'non_informative_prior',
                },
                'iii': {
                    'source': 'perfect_model',
                    'prior': 'informative_prior',
                },
                'iv': {
                    'source': 'perfect_model',
                    'prior': 'identity_transform',
                },
                'v': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'likelihood',
                },
                'vi': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'non_informative_prior',
                },
                'vii': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'informative_prior',
                },
                'viii': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'identity_transform',
                },
                'ix': {
                    'source': 'largely_misspecified_model',
                    'prior': 'likelihood',
                },
                'x': {
                    'source': 'largely_misspecified_model',
                    'prior': 'non_informative_prior',
                },
                'xi': {
                    'source': 'largely_misspecified_model',
                    'prior': 'informative_prior',
                },
                'xii': {
                    'source': 'largely_misspecified_model',
                    'prior': 'identity_transform',
                },
            }
        }
    }
else:
    with (experiment_dir/'execution_config.pkl').open('rb') as f:
        exec_configs = dill.load(f)

pprint.pp(exec_configs)

{'general': {'experiment_name': 'test_async_io',
             'experiment_dir': PosixPath('output/test_async_io'),
             'create_folder_structure': True,
             'n_sources': 2,
             'n_obs': 1000,
             'n_workers': 8,
             'n_realizations': 20,
             'A': array([[ 1. ,  1. ],
       [-0.5,  0.5]]),
             'B': array([[ 0.5, -1. ],
       [ 0.5,  1. ]]),
             'initial_B': array([[[ 0.67367376],
        [-0.81020861]],

       [[ 0.28932266],
        [ 0.98510879]]]),
             'normalize_posterior': True},
 'contour': {'contour_grid_points': 11,
             'u_lims': (-0.3, 0.3),
             'v_lims': (-0.3, 0.3)},
 'map': {'stopping_thresh': 0.0001, 'max_it': 100, 'learning_rate': 0.001},
 'mcmc': {'exploration_var': 5e-05, 'n_samples': 500, 'burn_in': 0.5},
 'sim': {'source_model': <src.source.LogisticSource object at 0x1119a57f0>,
         'sources': {'perfect_model': <src.source.LogisticSource object at 0x105dcc910>,
   

# 2. Folder Structure

In [5]:
# Initialize folder structure, if so specified
if CREATE_FOLDER_STRUCTURE:
    if exec_configs['general']['create_folder_structure']:
        # Creates base output path and experiment dir
        if not base_output_path.is_dir():
            base_output_path.mkdir()
        if not experiment_dir.is_dir():
            experiment_dir.mkdir()
        # Creates folders for individual realizations
        for r in range(exec_configs['general']['n_realizations']):
            # Overall folder for realization
            realization_dir = experiment_dir / str(r)
            if not realization_dir.is_dir():
                realization_dir.mkdir()
            # Initialize success flag
            with (realization_dir/'success_flag.pkl').open('wb') as f:
                    dill.dump('UNFINISHED', f)
        

if CREATE_CONFIG:
     with (experiment_dir/'execution_config.pkl').open('wb') as f:
        dill.dump(exec_configs, f)


# 3. Execute Experiment

In [6]:
ex = ExperimentExecutor(
    cfg=exec_configs,
    initialize=CREATE_CONFIG
)

In [7]:
%%time
ex.run()

> /Users/leonardo.tessarolo/git/bayesian_bss/src/executor.py(273)__get_iterable()
    271 
    272             if success_flag=='SUCCESS':
--> 273                 import pdb; pdb.set_trace()
    274                 finished_realizations.append(success_flag)
    275 

> /Users/leonardo.tessarolo/git/bayesian_bss/src/executor.py(273)__get_iterable()
    271 
    272             if success_flag=='SUCCESS':
--> 273                 import pdb; pdb.set_trace()
    274                 finished_realizations.append(success_flag)
    275 

1
> /Users/leonardo.tessarolo/git/bayesian_bss/src/executor.py(273)__get_iterable()
    271 
    272             if success_flag=='SUCCESS':
--> 273                 import pdb; pdb.set_trace()
    274                 finished_realizations.append(success_flag)
    275 

> /Users/leonardo.tessarolo/git/bayesian_bss/src/executor.py(273)__get_iterable()
    271 
    272             if success_flag=='SUCCESS':
--> 273                 import pdb; pdb.set_trace()
   

BdbQuit: 